In [1]:
#Install these packages before running the code

!conda install -c conda-forge gdal -y
!conda install rasterio
!conda install boto3
!conda install botocore

Channels:
 - conda-forge
Platform: linux-64
Solving environment: - ^C
^C

CondaError: KeyboardInterrupt

^C

CondaError: KeyboardInterrupt

^C

CondaError: KeyboardInterrupt



In [38]:
#imports
import os
import numpy as np
import rasterio
from rasterio.mask import mask
from collections import defaultdict
import logging
import boto3
import botocore
from pathlib import Path
import re
import shutil
import time
from datetime import timedelta
from osgeo import gdal

In [28]:
# ---------------------------------------------------------------------------
# Paths — match download_from_copernicus notebook
# ---------------------------------------------------------------------------

BASE = Path("/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data")

# Copernicus download writes SAFE products here:
#   new_data/{YYYY-MM-DD}/{S2*_MSIL2A_*}/
INPUT_DIR = BASE / "new_data"

# Stacked GeoTIFF outputs (per date):
#   new_data/{YYYY-MM-DD}/preprocessed/
def preprocessed_dir(date_str: str) -> Path:
    return INPUT_DIR / date_str / "preprocessed"

# Same bands as download_from_copernicus
BANDS = [
    "B02", "B03", "B04",
    "B05", "B06", "B07",
    "B08", "B11", "B12",
]

# Preferred resolution folder inside L2A SAFE products
BAND_RES = {
    "B02": "10m", "B03": "10m", "B04": "10m", "B08": "10m",
    "B05": "20m", "B06": "20m", "B07": "20m", "B11": "20m", "B12": "20m",
}

# Local scratch (temp work only)
WORK_INPUT_DIR = Path(os.getcwd()) / "input_dir"
WORK_OUTPUT_DIR = Path(os.getcwd()) / "output_dir"


def create_folder(folder_path):
    folder_path = Path(folder_path)
    if not folder_path.exists():
        folder_path.mkdir(parents=True, exist_ok=True)
        print(f"Folder '{folder_path}' created.")
    else:
        print(f"Folder '{folder_path}' already exists.")


create_folder(WORK_INPUT_DIR)
create_folder(WORK_OUTPUT_DIR)

print("BASE      :", BASE)
print("INPUT_DIR :", INPUT_DIR,
      "(exists)" if INPUT_DIR.is_dir() else "(not found — run Copernicus download first)")
print("OUTPUT    : new_data/{YYYY-MM-DD}/preprocessed/")


Folder '/home/jovyan/data-store/SCE-CU/Mesma/preprocessing/input_dir' already exists.
Folder '/home/jovyan/data-store/SCE-CU/Mesma/preprocessing/output_dir' already exists.
BASE      : /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data
INPUT_DIR : /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data (exists)
OUTPUT    : new_data/{YYYY-MM-DD}/preprocessed/


In [29]:
# ---------------------------------------------------------------------------
# Discover SAFE products + band JP2s from Copernicus download layout
# ---------------------------------------------------------------------------

def list_safe_products(input_dir: Path):
    """Return list of (date_str, product_dir) under new_data/{date}/{product}."""
    products = []
    if not input_dir.is_dir():
        return products

    for date_dir in sorted(input_dir.iterdir()):
        if not date_dir.is_dir() or date_dir.name.startswith("."):
            continue
        for product_dir in sorted(date_dir.iterdir()):
            if not product_dir.is_dir() or product_dir.name.startswith("."):
                continue
            # Skip the per-date output folder
            if product_dir.name == "preprocessed":
                continue
            products.append((date_dir.name, product_dir))
    return products


def find_band_jp2(product_dir: Path, band: str):
    """Find one JP2 for a band inside a SAFE / product tree."""
    preferred = BAND_RES[band]
    candidates = []

    for root, _, files in os.walk(product_dir):
        for name in files:
            if not name.lower().endswith(".jp2"):
                continue
            # L2A style: ..._B02_10m.jp2  or older ..._B02.jp2
            if f"_{band}_" in name or name.endswith(f"_{band}.jp2"):
                candidates.append(Path(root) / name)

    if not candidates:
        return None

    # Prefer matching resolution (10m / 20m)
    preferred_hits = [p for p in candidates if preferred in p.name]
    if preferred_hits:
        return preferred_hits[0]
    return candidates[0]


def product_output_name(product_dir: Path, date_str: str) -> str:
    """
    Build MESMA-style name: 10_S_EG_2026_7_28.tif
    from product id like S2A_MSIL2A_..._T10SEG_...
    """
    name = product_dir.name
    # Tile token e.g. T10SEG
    m = re.search(r"_T([0-9]{2}[A-Z]{3})_", name)
    if m:
        tile = m.group(1)  # 10SEG
        utm, lat, square = tile[:2], tile[2], tile[3:]
        tile_part = f"{utm}_{lat}_{square}"
    else:
        tile_part = "UNKNOWN"

    # Prefer sensing date from product id YYYYMMDD
    m = re.search(r"_(\d{8})T", name)
    if m:
        ymd = m.group(1)
        y, mo, d = int(ymd[:4]), int(ymd[4:6]), int(ymd[6:8])
        date_part = f"{y}_{mo}_{d}"
    else:
        # fallback: folder date YYYY-MM-DD
        y, mo, d = date_str.split("-")
        date_part = f"{int(y)}_{int(mo)}_{int(d)}"

    return f"{tile_part}_{date_part}.tif"


def copy_band_to_work(src: Path, band: str, dest_dir: Path) -> Path:
    dest = dest_dir / f"{band}.jp2"
    print("Copy:", src, "->", dest)
    shutil.copy2(src, dest)
    return dest


In [30]:
# ---------------------------------------------------------------------------
# Resample + stack (same idea as original preprocessing)
# ---------------------------------------------------------------------------

def delete_files(files):
    for file in files:
        path = Path(file)
        if path.is_file():
            path.unlink()


def resample_bands(work_dir: Path):
    print("Resampling 20 m bands to 10 m")
    bands_to_resample = ["B05", "B06", "B07", "B11", "B12"]
    for band in bands_to_resample:
        file_path = work_dir / f"{band}.jp2"
        if file_path.is_file():
            print("Resampling", file_path)
            gdal.Warp(str(file_path), str(file_path), xRes=10, yRes=10)


def stack_bands(band_files, output_file: Path):
    print("Stacking bands ->", output_file)
    band_arrays = []
    meta = None
    band_crs = None

    for idx, band_file in enumerate(band_files):
        with rasterio.open(band_file, driver="JP2OpenJPEG") as src:
            band_arrays.append(src.read(1))
            band_crs = src.crs
            if idx == 0:
                meta = src.meta.copy()

    meta.update({
        "driver": "GTiff",
        "count": len(band_arrays),
        "dtype": band_arrays[0].dtype,
        "crs": band_crs,
    })

    with rasterio.open(output_file, "w", **meta) as dst:
        for i, band_array in enumerate(band_arrays, start=1):
            dst.write(band_array, indexes=i)

    print(f"Stacked GeoTIFF written to {output_file}")


def save_to_preprocessed(local_output_file: Path, dest_dir: Path) -> Path:
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / local_output_file.name
    print("Write to preprocessed:", dest)
    shutil.copy2(local_output_file, dest)
    return dest

In [31]:
# # ---------------------------------------------------------------------------
# # Main preprocessing for Copernicus SAFE products
# # ---------------------------------------------------------------------------

# def process_images(input_dir: Path = INPUT_DIR):
#     """
#     Preprocess Copernicus L2A products from new_data/{date}/{product}/
#     into stacked GeoTIFFs under new_data/{date}/preprocessed/.
#     """
#     create_folder(WORK_INPUT_DIR)
#     create_folder(WORK_OUTPUT_DIR)

#     if not input_dir.is_dir():
#         print(f"Skip — input not found: {input_dir}")
#         print("  Run download_from_copernicus first so new_data/{date}/{product}/ exists.")
#         return

#     products = list_safe_products(input_dir)
#     error_files = []

#     print("INPUT_DIR :", input_dir)
#     print("OUTPUT    : new_data/{YYYY-MM-DD}/preprocessed/")
#     print("Products found:", len(products))

#     for i, (date_str, product_dir) in enumerate(products, 1):
#         output_dir = preprocessed_dir(date_str)
#         create_folder(output_dir)

#         existing_outputs = {p.name for p in output_dir.glob("*.tif")}
#         output_name = product_output_name(product_dir, date_str)
#         print(f"\n[{i}/{len(products)}] {product_dir.name} -> {output_dir / output_name}")

#         if output_name in existing_outputs:
#             print("Already exists, skip:", output_name)
#             continue

#         band_srcs = {}
#         missing = []
#         for band in BANDS:
#             jp2 = find_band_jp2(product_dir, band)
#             if jp2 is None:
#                 missing.append(band)
#             else:
#                 band_srcs[band] = jp2

#         if missing:
#             print(f"Skipping — missing bands {missing}")
#             error_files.append(output_name)
#             continue

#         # Clear work dir for this product
#         for old in WORK_INPUT_DIR.glob("*"):
#             if old.is_file():
#                 old.unlink()

#         work_band_paths = []
#         local_output = WORK_OUTPUT_DIR / output_name

#         try:
#             for band in BANDS:
#                 work_band_paths.append(
#                     copy_band_to_work(band_srcs[band], band, WORK_INPUT_DIR)
#                 )

#             resample_bands(WORK_INPUT_DIR)
#             stack_bands(work_band_paths, local_output)
#             save_to_preprocessed(local_output, output_dir)

#         except Exception as e:
#             print("Error processing", output_name, ":", e)
#             error_files.append(output_name)

#         delete_files(work_band_paths + [local_output])

#     print("\nDone. Products visited:", len(products), "| Errors:", len(error_files))
#     if error_files:
#         print("Error / skipped outputs:", error_files)


In [32]:
from datetime import datetime

# ---------------------------------------------------------------------------
# Main preprocessing for Copernicus SAFE products
# ---------------------------------------------------------------------------

def process_images(input_dir: Path = INPUT_DIR):
    """
    Preprocess Copernicus L2A products from:

        new_data/{date}/{product}.SAFE

    into

        new_data/{date}/preprocessed/*.tif
    """

    create_folder(WORK_INPUT_DIR)
    create_folder(WORK_OUTPUT_DIR)

    if not input_dir.is_dir():
        print(f"Input directory not found:\n{input_dir}")
        return

    products = list_safe_products(input_dir)

    total = len(products)

    completed = 0
    skipped = 0
    failed = 0

    error_files = []

    print("=" * 80)
    print("Sentinel-2 Image Preprocessing")
    print("=" * 80)
    print(f"Products found : {total}")

    overall_start = datetime.now()

    for i, (date_str, product_dir) in enumerate(products, start=1):

        percent = 100 * i / total

        print("\n" + "=" * 80)
        print(f"[{i}/{total}] ({percent:5.1f}%)")
        print(product_dir.name)
        print("=" * 80)

        start = datetime.now()

        output_dir = preprocessed_dir(date_str)
        create_folder(output_dir)

        output_name = product_output_name(product_dir, date_str)

        output_file = output_dir / output_name

        if output_file.exists():

            print("✅ Already processed")

            skipped += 1
            continue

        band_srcs = {}
        missing = []

        for band in BANDS:

            jp2 = find_band_jp2(product_dir, band)

            if jp2 is None:
                missing.append(band)
            else:
                band_srcs[band] = jp2

        if missing:

            print(f"❌ Missing bands: {missing}")

            failed += 1
            error_files.append(output_name)

            continue

        for old in WORK_INPUT_DIR.glob("*"):
            if old.is_file():
                old.unlink()

        work_band_paths = []

        local_output = WORK_OUTPUT_DIR / output_name

        try:

            print("Copying bands...")

            for band in BANDS:

                work_band_paths.append(
                    copy_band_to_work(
                        band_srcs[band],
                        band,
                        WORK_INPUT_DIR,
                    )
                )

            print("Resampling...")
            resample_bands(WORK_INPUT_DIR)

            print("Stacking...")
            stack_bands(work_band_paths, local_output)

            print("Saving...")
            save_to_preprocessed(local_output, output_dir)

            completed += 1

            print(f"✅ Finished in {datetime.now()-start}")

        except Exception as e:

            failed += 1

            print("❌ ERROR:", e)

            error_files.append(output_name)

        finally:

            delete_files(work_band_paths + [local_output])

    print("\n" + "=" * 80)
    print("Preprocessing finished")
    print("=" * 80)

    print(f"Processed : {completed}")
    print(f"Skipped   : {skipped}")
    print(f"Failed    : {failed}")
    print(f"Elapsed   : {datetime.now()-overall_start}")

    if error_files:

        print("\nFailed products:")

        for f in error_files:
            print(" -", f)

In [33]:
# import re
# import time
# from datetime import timedelta

# RUN = True

# print("=" * 80)
# print("Sentinel-2 Preprocessing")
# print("=" * 80)

# print(f"Input : {INPUT_DIR}")
# print(f"Output: {INPUT_DIR}/{{YYYY-MM-DD}}/preprocessed")

# start = time.time()

# if RUN:
#     process_images(INPUT_DIR)
# else:
#     print("RUN=False — preprocessing skipped.")

# elapsed = timedelta(seconds=int(time.time() - start))

# print("\n" + "=" * 80)
# print("Finished")
# print(f"Total runtime: {elapsed}")
# print("=" * 80)

In [34]:
# import re
# # ---------------------------------------------------------------------------
# # Run preprocessing
# #
# #   1) download_from_copernicus -> new_data/{YYYY-MM-DD}/{S2*_MSIL2A_*}/
# #   2) this notebook            -> new_data/{YYYY-MM-DD}/preprocessed/*.tif
# #   3) MESMA
# # ---------------------------------------------------------------------------

# RUN = True  # set False to only print paths

# if RUN:
#     process_images(INPUT_DIR)
# else:
#     print("RUN=False — not preprocessing.")

# print("\nConfigured paths:")
# print("  input :", INPUT_DIR / "{YYYY-MM-DD}" / "{S2*_MSIL2A_*}")
# print("  output:", INPUT_DIR / "{YYYY-MM-DD}" / "preprocessed")


In [39]:


# ---------------------------------------------------------------------------
# Run preprocessing
#
#   1) download_from_copernicus -> new_data/{YYYY-MM-DD}/{S2*_MSIL2A_*.SAFE}
#   2) preprocess               -> new_data/{YYYY-MM-DD}/preprocessed/*.tif
#   3) MESMA
# ---------------------------------------------------------------------------

RUN = True

print("=" * 80)
print("Sentinel-2 preprocessing")
print("=" * 80)

print(f"Input : {INPUT_DIR}")
print(f"Output: {INPUT_DIR}/{{YYYY-MM-DD}}/preprocessed")

start = time.time()

if RUN:
    print("\nStarting preprocessing...\n")
    process_images(INPUT_DIR)
else:
    print("\nRUN=False -- preprocessing skipped.")

elapsed = timedelta(seconds=int(time.time() - start))

print("\n" + "=" * 80)
print("Finished.")
print(f"Elapsed time: {elapsed}")
print("=" * 80)

Sentinel-2 preprocessing
Input : /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data
Output: /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/{YYYY-MM-DD}/preprocessed

Starting preprocessing...

Folder '/home/jovyan/data-store/SCE-CU/Mesma/preprocessing/input_dir' already exists.
Folder '/home/jovyan/data-store/SCE-CU/Mesma/preprocessing/output_dir' already exists.
Sentinel-2 Image Preprocessing
Products found : 49

[1/49] (  2.0%)
S2A_MSIL2A_20260728T183831_N0512_R027_T10SGB_20260729T025553.SAFE
Folder '/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/preprocessed' already exists.
Copying bands...
Copy: /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/S2A_MSIL2A_20260728T183831_N0512_R027_T10SGB_20260729T025553.SAFE/GRANULE/L2A_T10SGB_A057968_20260728T183828/IMG_DATA/R10m/T10SGB_20260728T183831_B02_10m.jp2 -> /home/jovyan/data-store/SCE-CU/Mesma/preprocessing/input_dir/B02.jp2
Copy: /data-store/ipl

/opt/conda/envs/macrosystems/lib/python3.10/site-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Resampling /home/jovyan/data-store/SCE-CU/Mesma/preprocessing/input_dir/B06.jp2
Resampling /home/jovyan/data-store/SCE-CU/Mesma/preprocessing/input_dir/B07.jp2
Resampling /home/jovyan/data-store/SCE-CU/Mesma/preprocessing/input_dir/B11.jp2
Resampling /home/jovyan/data-store/SCE-CU/Mesma/preprocessing/input_dir/B12.jp2
Stacking...
Stacking bands -> /home/jovyan/data-store/SCE-CU/Mesma/preprocessing/output_dir/10_S_GB_2026_7_28.tif
Stacked GeoTIFF written to /home/jovyan/data-store/SCE-CU/Mesma/preprocessing/output_dir/10_S_GB_2026_7_28.tif
Saving...
Write to preprocessed: /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/preprocessed/10_S_GB_2026_7_28.tif
✅ Finished in 0:07:17.520413

[2/49] (  4.1%)
S2A_MSIL2A_20260728T183831_N0512_R027_T11SKR_20260729T025553.SAFE
Folder '/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/preprocessed' already exists.
Copying bands...
Copy: /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new